### This project's workflow and objectives are as follows:

- Create a "Text==> Vader Pipeline"
- Cretae a "Text ==> Remove Stopwords ==> Vader Pipeline"
- Create a "Text ==> Remove Stopwords ==>  Bag Of Words ==> Custom Model"
- Create a "Text ==> Remove Stopwords ==>  TF-IDF ==> Custom Model"

### In order to accomplish this we neeed to perfoem:

- Data Collection
- Text Processing
- Modelling
- Model Evaluation

### Data Loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings

<function warnings.filterwarnings(action, message='', category=<class 'Warning'>, module='', lineno=0, append=False)>

In [2]:
df = pd.read_csv(r"C:\Users\USER\Desktop\Wahab Folder\Data science\My Projects\Sentiment Analysis\Deploymnent with Streamlit\data.csv")

In [3]:
df.head()

,Unnamed: 0,Username,Location,Total Review,Date of Experience,Content,Rating
0,0,Christopher Smith,GB,3reviews,"January 30, 2024",I have been shopping with AliExpress for some ...,4
1,1,blarp.tha.alien,AU,1review,"May 16, 2024",AliExpress is Legit a great place to find amaz...,5
2,2,Margarita Chavez Villalobos,US,3reviews,"May 13, 2024",AliExpress does pretty good on delivering SMAL...,3
3,3,Mario Alzate,US,1review,"May 18, 2024","Well, if you educate yourself just a little on...",5
4,4,AE user,US,1review,"May 04, 2024","Found some nice deals, but also ran into a sca...",5


In [4]:
# rename the Unnamed:0 column to id
df.columns = ['id', 'username', 'location', 'total review', 'date of experience', 'text', 'rating']
df.head(3)

,id,username,location,total review,date of experience,text,rating
0,0,Christopher Smith,GB,3reviews,"January 30, 2024",I have been shopping with AliExpress for some ...,4
1,1,blarp.tha.alien,AU,1review,"May 16, 2024",AliExpress is Legit a great place to find amaz...,5
2,2,Margarita Chavez Villalobos,US,3reviews,"May 13, 2024",AliExpress does pretty good on delivering SMAL...,3


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46100 entries, 0 to 46099
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  46100 non-null  int64 
 1   username            46096 non-null  object
 2   location            46088 non-null  object
 3   total review        46100 non-null  object
 4   date of experience  46100 non-null  object
 5   text                36717 non-null  object
 6   rating              46100 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 2.5+ MB


In [6]:
# create a copy of the dataset
df1 = df.copy()

In [7]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46100 entries, 0 to 46099
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  46100 non-null  int64 
 1   username            46096 non-null  object
 2   location            46088 non-null  object
 3   total review        46100 non-null  object
 4   date of experience  46100 non-null  object
 5   text                36717 non-null  object
 6   rating              46100 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 2.5+ MB


In [8]:
df1.isnull().sum()

id                       0
username                 4
location                12
total review             0
date of experience       0
text                  9383
rating                   0
dtype: int64

In [9]:
# To see only row with the missing data
df1[df1.isnull().any(axis=1)]

,id,username,location,total review,date of experience,text,rating
139,139,Kamal Seth,US,3reviews,"May 15, 2024",NaN,5
140,140,Darlene Pino,US,3reviews,"May 01, 2024",NaN,5
144,144,Dalibor Blazek,CZ,1review,"May 02, 2024",NaN,5
150,150,Camy Liu,KH,1review,"May 16, 2024",NaN,5
155,155,michele,US,1review,"May 05, 2024",NaN,5
...,...,...,...,...,...,...,...
44246,44246,Sergey Borkovsky,RU,1review,"September 07, 2016",NaN,5
44247,44247,ALEXANDRE PIRES DE CAMARGO,BR,1review,"September 07, 2016",NaN,5
44745,44745,Jinliang Liu,CN,1review,"December 17, 2015",NaN,5
45790,45790,Сергей,RU,1review,"April 19, 2014",NaN,5


In [10]:
df1.dropna(inplace = True)

In [11]:
df1.isnull().sum()

id                    0
username              0
location              0
total review          0
date of experience    0
text                  0
rating                0
dtype: int64

In [12]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36706 entries, 0 to 46099
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  36706 non-null  int64 
 1   username            36706 non-null  object
 2   location            36706 non-null  object
 3   total review        36706 non-null  object
 4   date of experience  36706 non-null  object
 5   text                36706 non-null  object
 6   rating              36706 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 2.2+ MB


In [13]:
# checking the review in column 0
df1.iloc[0]

id                                                                    0
username                                              Christopher Smith
location                                                             GB
total review                                                   3reviews
date of experience                                     January 30, 2024
text                  I have been shopping with AliExpress for some ...
rating                                                                4
Name: 0, dtype: object

In [14]:
# checking the full review in column 0
df1.iloc[0]['text']

'I have been shopping with AliExpress for some time now. I only ever chose "Choice" deliveries because I don\'t want to wait 4 to 6 weeks for my delivery. If you filter by "Choice" deliveries you should receive your item in about 7-8 days. I\'ve only had a problem with 1 seller who never delivered my item. AliExpress took their time resolving the issue, so they miss out on a star for that.'

In [15]:
df1.rating.unique()

array([4, 5, 3, 1, 2], dtype=int64)

In [16]:
df1["rating"].value_counts()

rating
5    22553
1    10433
4     1895
3      957
2      868
Name: count, dtype: int64

In [17]:
# mapping the Rating to sentiment words (positive, negative)

mapped_values = {
    1 : "negative",
    2 : "negative",
    3 : "neutral",
    4 : "positive",
    5 : "positive"
}

In [18]:
# mapping the rating column and applying on the dataset
df1['rating'] = df1['rating'].map(mapped_values)

In [19]:
df1.head()

,id,username,location,total review,date of experience,text,rating
0,0,Christopher Smith,GB,3reviews,"January 30, 2024",I have been shopping with AliExpress for some ...,positive
1,1,blarp.tha.alien,AU,1review,"May 16, 2024",AliExpress is Legit a great place to find amaz...,positive
2,2,Margarita Chavez Villalobos,US,3reviews,"May 13, 2024",AliExpress does pretty good on delivering SMAL...,neutral
3,3,Mario Alzate,US,1review,"May 18, 2024","Well, if you educate yourself just a little on...",positive
4,4,AE user,US,1review,"May 04, 2024","Found some nice deals, but also ran into a sca...",positive


In [20]:
df1["rating"].value_counts()

rating
positive    24448
negative    11301
neutral       957
Name: count, dtype: int64

### Text Processing
- Remove Stopwords
- Bag of words
- TF-IDF

In [21]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
#nltk requires this package to be installed alongside stopwords
nltk.download('punkt_tab')

stop_words = stopwords.words("english") 

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [22]:
# checking for the first 10 stop_words
stop_words[0:10]

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']

In [23]:
# Example sentence
text = "This is an example sentence with some stopwords that we want to remove, i love this product, it is good."

In [24]:
# tokenize the sentence: split sentence into list of words
words = nltk.word_tokenize(text)
words

['This',
 'is',
 'an',
 'example',
 'sentence',
 'with',
 'some',
 'stopwords',
 'that',
 'we',
 'want',
 'to',
 'remove',
 ',',
 'i',
 'love',
 'this',
 'product',
 ',',
 'it',
 'is',
 'good',
 '.']

In [25]:
# Remove stopwords from the text
filtered_words = [word for word in words if word.lower() not in stop_words]       #this returns a list
filtered_words

['example',
 'sentence',
 'stopwords',
 'want',
 'remove',
 ',',
 'love',
 'product',
 ',',
 'good',
 '.']

In [26]:
# Reconstruct the text in list to a string using the join function
filtered_text = " ".join(filtered_words)                                          #this returns the list back to a string, coz string is the accepted format
print(filtered_text)

example sentence stopwords want remove , love product , good .


In [27]:
# Passing the codes into a single funtion
def remove_stopwords(text):
  """
  this function takes in a sentence
  tokenize the sentence
  remove stopwords and 
  return the sentence
  """

  # tokenize the sentence: split sentence into list of words
  words = nltk.word_tokenize(text)
  # Remove stopwords from the text
  filtered_words = [word for word in words if word.lower() not in stop_words]
  # Reconstruct the text in list to a string using the join function
  filtered_text = " ".join(filtered_words)

  #print(filtered_text)
  return filtered_text


In [28]:
remove_stopwords(text)

'example sentence stopwords want remove , love product , good .'

### Applying the function on the dataset (df1)

In [29]:
len(df1)

36706

In [30]:
##i would love to see a progress bar when we process for all the reviews
total_rows = len(df1)
tqdm.pandas(total=total_rows)
df1['text_without_stopwords'] = df1['text'].progress_apply(remove_stopwords)

100%|███████████████████████████████████████████████████████████████████████████| 36706/36706 [00:37<00:00, 987.54it/s]


In [31]:
df1.head()

,id,username,location,total review,date of experience,text,rating,text_without_stopwords
0,0,Christopher Smith,GB,3reviews,"January 30, 2024",I have been shopping with AliExpress for some ...,positive,shopping AliExpress time . ever chose `` Choic...
1,1,blarp.tha.alien,AU,1review,"May 16, 2024",AliExpress is Legit a great place to find amaz...,positive,AliExpress Legit great place find amazing barg...
2,2,Margarita Chavez Villalobos,US,3reviews,"May 13, 2024",AliExpress does pretty good on delivering SMAL...,neutral,AliExpress pretty good delivering SMALL items ...
3,3,Mario Alzate,US,1review,"May 18, 2024","Well, if you educate yourself just a little on...",positive,"Well , educate little get best benefits search..."
4,4,AE user,US,1review,"May 04, 2024","Found some nice deals, but also ran into a sca...",positive,"Found nice deals , also ran scam ( fake item )..."


##### Split the data into training and testing sets

In [32]:
# Label
label = df1.pop('rating')

In [33]:
df1.head()

,id,username,location,total review,date of experience,text,text_without_stopwords
0,0,Christopher Smith,GB,3reviews,"January 30, 2024",I have been shopping with AliExpress for some ...,shopping AliExpress time . ever chose `` Choic...
1,1,blarp.tha.alien,AU,1review,"May 16, 2024",AliExpress is Legit a great place to find amaz...,AliExpress Legit great place find amazing barg...
2,2,Margarita Chavez Villalobos,US,3reviews,"May 13, 2024",AliExpress does pretty good on delivering SMAL...,AliExpress pretty good delivering SMALL items ...
3,3,Mario Alzate,US,1review,"May 18, 2024","Well, if you educate yourself just a little on...","Well , educate little get best benefits search..."
4,4,AE user,US,1review,"May 04, 2024","Found some nice deals, but also ran into a sca...","Found nice deals , also ran scam ( fake item )..."


In [34]:
label.head()

0    positive
1    positive
2     neutral
3    positive
4    positive
Name: rating, dtype: object

##### Data Balancing - Random Oversampling

In [35]:
df1.head()

,id,username,location,total review,date of experience,text,text_without_stopwords
0,0,Christopher Smith,GB,3reviews,"January 30, 2024",I have been shopping with AliExpress for some ...,shopping AliExpress time . ever chose `` Choic...
1,1,blarp.tha.alien,AU,1review,"May 16, 2024",AliExpress is Legit a great place to find amaz...,AliExpress Legit great place find amazing barg...
2,2,Margarita Chavez Villalobos,US,3reviews,"May 13, 2024",AliExpress does pretty good on delivering SMAL...,AliExpress pretty good delivering SMALL items ...
3,3,Mario Alzate,US,1review,"May 18, 2024","Well, if you educate yourself just a little on...","Well , educate little get best benefits search..."
4,4,AE user,US,1review,"May 04, 2024","Found some nice deals, but also ran into a sca...","Found nice deals , also ran scam ( fake item )..."


In [36]:
df1.shape

(36706, 7)

In [37]:
label.shape

(36706,)

In [38]:
label.value_counts()

rating
positive    24448
negative    11301
neutral       957
Name: count, dtype: int64

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df1, label, test_size=0.2, random_state=42)

In [40]:
X_train.head()

,id,username,location,total review,date of experience,text,text_without_stopwords
4655,4655,Guy Shiloh,IL,1review,"May 11, 2023",Communicating with this Seller via AliExpress'...,Communicating Seller via AliExpress ' messenge...
36398,36398,Sergey Semenov,RU,3reviews,"April 16, 2018","Large selection of products, low prices.","Large selection products , low prices ."
15135,15135,Mr.P,TH,1review,"April 28, 2020",Very Good Product.,Good Product .
17752,17752,Jumpot Therajindachol,TH,1review,"January 25, 2020","Good products as ordered, but transportation i...","Good products ordered , transportation delayed ."
11499,11499,Diogo Leone Estevam,BR,37reviews,"September 08, 2020","Seller, mediation and product absolutely usell...","Seller , mediation product absolutely uselles ..."


In [41]:
y_train.head()

4655     positive
36398    positive
15135    positive
17752     neutral
11499    positive
Name: rating, dtype: object

In [42]:
X_train.iloc[17752]['text']

"make a bit of money. Customer service is very questionable and representatives often lie to you. AliExpress think they can make their own rules/laws and don't act according to international laws. If you get what you paid for you can have a great deal. But if you don't, you are completely on your own.Aliexpress 𝕔𝕦𝕤𝕥𝕠𝕞𝕖𝕣 𝕤𝕖𝕣𝕧𝕚𝕔𝕖 𝕙𝕖𝕝𝕡𝕝𝕚𝕟𝕖 𝕥𝕠𝕝𝕝-𝕗𝕣𝕖𝕖 𝕡𝕙𝕠𝕟𝕖 𝕟𝕦𝕞𝕓𝕖𝕣 𝕚𝕤 +𝟙 ''𝟠𝟝''**'''''𝟝''𝟛'𝟞''𝟡'''𝟠'''𝟝'𝟞'''''***𝟝Aliexpress 𝕔𝕦𝕤𝕥𝕠𝕞𝕖𝕣 𝕤𝕖𝕣𝕧𝕚𝕔𝕖 𝕙𝕖𝕝𝕡𝕝𝕚𝕟𝕖 𝕥𝕠𝕝𝕝-𝕗𝕣𝕖𝕖 𝕡𝕙𝕠𝕟𝕖 𝕟𝕦𝕞𝕓𝕖𝕣 𝕚𝕤 +𝟙 ''𝟠𝟝''**'''''𝟝''𝟛'𝟞''𝟡'''𝟠'''𝟝'𝟞'''''***𝟝Aliexpress 𝕔𝕦𝕤𝕥𝕠𝕞𝕖𝕣 𝕤𝕖𝕣𝕧𝕚𝕔𝕖 𝕙𝕖𝕝𝕡𝕝𝕚𝕟𝕖 𝕥𝕠𝕝𝕝-𝕗𝕣𝕖𝕖 𝕡𝕙𝕠𝕟𝕖 𝕟𝕦𝕞𝕓𝕖𝕣 𝕚𝕤 +𝟙 ''𝟠𝟝''**'''''𝟝''𝟛'𝟞''𝟡'''𝟠'''𝟝'𝟞'''''***𝟝make a bit of money. Customer service is very questionable and representatives often lie to you. AliExpress think they can make their own rules/laws and don't act according to international laws. If you get what you paid for you can have a great deal. But if you don't, you are completely on your own."

In [43]:
X_test.head()

,id,username,location,total review,date of experience,text,text_without_stopwords
45973,45973,Paul Philps,CA,3reviews,"August 31, 2013",I contacted a company about making 17 hockey j...,contacted company making 17 hockey jerseys ice...
22287,22287,TSANG David,FR,1review,"October 04, 2019","Received very quickly, I recommend for this se...","Received quickly , recommend seller , good pro..."
34019,34019,Sunil,CH,1review,"July 20, 2018",I have been using Aliexpress for years. They d...,"using Aliexpress years . help , sellers want m..."
16966,16966,Mikola Dmitruk,UA,1review,"February 19, 2020",Everything works flawlessly. Thank you.,Everything works flawlessly . Thank .
30467,30467,Joe Higashi,PH,1review,"January 10, 2019",their warm greetings makes I feel comfortable....,warm greetings makes feel comfortable . Ive st...


In [44]:
y_test.head()

45973    negative
22287    positive
34019    positive
16966    positive
30467    positive
Name: rating, dtype: object

##### CREATING BAG OF WORDS (BoW) AND TF-IDF (tf-idf)
A "Bag of Words" (BoW) is a simple and commonly used technique in natural language processing (NLP) and text analysis to represent text data as numerical features. It is used to transform a collection of text documents into a format that can be processed by machine learning algorithms. The idea behind the Bag of Words model is to disregard the order and structure of words in a text and focus only on the frequency of each word's occurrence.

The key idea is that the order of words and the grammatical structure of sentences are ignored, and the analysis is purely based on the presence or absence of specific words and their frequencies.

TF-IDF, which stands for "Term Frequency-Inverse Document Frequency," is a numerical statistic used in information retrieval and natural language processing (NLP) to evaluate the importance of a word within a document relative to a collection of documents, typically a corpus.

The TF-IDF score provides a measure of how important a term is within a specific document and across a collection of documents. Terms that appear frequently in a document but rarely in other documents receive higher TF-IDF scores, making them indicative of the content of that document.

In [45]:
from sklearn.feature_extraction.text import CountVectorizer   # to create bagof words
from sklearn.feature_extraction.text import TfidfVectorizer   # to create tfidf

# now vectorize the oversampled (but still real) text
bow_vectorizer = CountVectorizer(ngram_range=(1, 2)) 
X_train_bow = bow_vectorizer.fit_transform(X_train['text_without_stopwords'])
X_test_bow = bow_vectorizer.transform(X_test['text_without_stopwords'])

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2)) 
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['text_without_stopwords'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['text_without_stopwords'])

### Data Balancing

In [46]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_bow_res, y_train_bow_res = ros.fit_resample(X_train_bow, y_train)
X_train_tfidf_res, y_train_tfidf_res = ros.fit_resample(X_train_tfidf, y_train)

print('Before oversampling:', y_train.value_counts().to_dict())
print('After oversampling: ', y_train_bow_res.value_counts().to_dict())

Before oversampling: {'positive': 19580, 'negative': 9000, 'neutral': 784}
After oversampling:  {'positive': 19580, 'neutral': 19580, 'negative': 19580}


##### MODELLING AND EVALUATION
- vader on normal sentences (text)
- vader on text_without_stopwords
- custom on train_tf-idf
- custom on train bow

##### Vader Model

In [47]:
import nltk
# download the VADER lexicon and model
nltk.download('vader_lexicon') 

[nltk_data] Error loading vader_lexicon: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [48]:
# next we import the SentimentIntensityAnalyzer class from vader
# Vader is a pretrained model for analysing sentiments of sentences
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [49]:
analyzer = SentimentIntensityAnalyzer()

In [50]:
# lets test out the sentiment analyzer with an example text
example_text  = "i love the orange flavor, good product"

In [51]:
# sentiment scores
sentiment_scores = analyzer.polarity_scores(example_text)
sentiment_scores

{'neg': 0.0, 'neu': 0.36, 'pos': 0.64, 'compound': 0.7964}

In [52]:
# compound score
compound_score = sentiment_scores['compound']
compound_score

0.7964

In [53]:
# example text2
example_text2  = "i don't like the orange flavor"

In [54]:
# sentiment scores
sentiment_scores2 = analyzer.polarity_scores(example_text2)
sentiment_scores2

{'neg': 0.345, 'neu': 0.655, 'pos': 0.0, 'compound': -0.2755}

In [55]:
# compound score
compound_score = sentiment_scores2['compound']
compound_score

-0.2755

In [56]:
#getting the sentiment scores
compound_score = sentiment_scores['compound']

# now lets make a decision for the cut off for a postitive or negative score
if compound_score >= 0.05:
    sentiment = "positive"
elif compound_score <= 0.05:
    sentiment = "negative"  
else:
    sentiment = "neutral"

print(f"The sentiment is {sentiment} (Compound Score: {compound_score})")

The sentiment is positive (Compound Score: 0.7964)


In [57]:
##we want to apply all we just did to all the text in our dataset, so lets first
##create the function, then we apply the function

def analyze_sentence(sentence):
  """
  - this function takes in a sentence and 
  - returns the sentiment using analyzer (returns positive if compound score is greater than threshold (compound_score > 0), else 
  returns negative).
  """
  sentiment_scores = analyzer.polarity_scores(sentence)
  compound_score = sentiment_scores['compound']

  

  if compound_score >= 0.05:
    return "positive"
  elif compound_score <= 0.05:
    return "negative"  
  else:
    return "neutral"


In [58]:
analyze_sentence(example_text)

'positive'

In [59]:
analyze_sentence(example_text2)

'negative'

##### 1. applying vader on text column on test dataset (X_test)  

In [60]:
X_test['vader_on_text'] = X_test['text'].apply(analyze_sentence)

##### 2. applying vader on text_without_stopwords column on test dataset (X_test)

In [61]:
# applying vader on text_without_stopwords
X_test['vader_on_text_without_stopwords'] = X_test['text_without_stopwords'].apply(analyze_sentence)

In [62]:
X_test.head()

,id,username,location,total review,date of experience,text,text_without_stopwords,vader_on_text,vader_on_text_without_stopwords
45973,45973,Paul Philps,CA,3reviews,"August 31, 2013",I contacted a company about making 17 hockey j...,contacted company making 17 hockey jerseys ice...,positive,positive
22287,22287,TSANG David,FR,1review,"October 04, 2019","Received very quickly, I recommend for this se...","Received quickly , recommend seller , good pro...",positive,positive
34019,34019,Sunil,CH,1review,"July 20, 2018",I have been using Aliexpress for years. They d...,"using Aliexpress years . help , sellers want m...",positive,positive
16966,16966,Mikola Dmitruk,UA,1review,"February 19, 2020",Everything works flawlessly. Thank you.,Everything works flawlessly . Thank .,positive,positive
30467,30467,Joe Higashi,PH,1review,"January 10, 2019",their warm greetings makes I feel comfortable....,warm greetings makes feel comfortable . Ive st...,positive,positive


##### Training custom models on bag of words and tf-idf

In [63]:
X_train_bow

<29364x394382 sparse matrix of type '<class 'numpy.int64'>'
	with 1280550 stored elements in Compressed Sparse Row format>

In [64]:
X_train_tfidf

<29364x394382 sparse matrix of type '<class 'numpy.float64'>'
	with 1280550 stored elements in Compressed Sparse Row format>

In [65]:
X_test_bow

<7342x394382 sparse matrix of type '<class 'numpy.int64'>'
	with 244569 stored elements in Compressed Sparse Row format>

In [66]:
X_test_tfidf

<7342x394382 sparse matrix of type '<class 'numpy.float64'>'
	with 244569 stored elements in Compressed Sparse Row format>

print(X_train_tfidf.toarray())

### 1. Multinomial Naive Bayes

In [67]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [68]:
from sklearn.naive_bayes import MultinomialNB

In [69]:
#create a classifier
mnb_bow = MultinomialNB()
mnb_tfidf = MultinomialNB()

In [70]:
#fit on bag_of_words & tf-idf
mnb_bow.fit(X_train_bow_res, y_train_bow_res)
mnb_tfidf.fit(X_train_tfidf_res, y_train_tfidf_res)

MultinomialNB()

In [71]:
## Making prediction on test_data (test_bow & test_tfidf)
X_test['mnb_bow'] = mnb_bow.predict(X_test_bow)
X_test['mnb_tfidf'] = mnb_tfidf.predict(X_test_tfidf)

### 2. Logistic Regression

In [72]:
from sklearn.linear_model import LogisticRegression

In [73]:
logreg_bow = LogisticRegression()
logreg_tfidf = LogisticRegression()

In [74]:
#fit on bag_of_words & tf-idf
logreg_bow.fit(X_train_bow_res, y_train_bow_res)
logreg_tfidf.fit(X_train_tfidf_res, y_train_tfidf_res)

LogisticRegression()

In [75]:
## Making prediction on test_data (test_bow & test_tfidf)
X_test['logreg_bow'] = logreg_bow.predict(X_test_bow)
X_test['logreg_tfidf'] = logreg_tfidf.predict(X_test_tfidf)

### 3. Linear SVM

In [76]:
from sklearn.svm import LinearSVC

In [77]:
svm_bow = LinearSVC()
svm_tfidf = LinearSVC()

In [78]:
#fit on bag_of_words & tf-idf
svm_bow.fit(X_train_bow_res, y_train_bow_res)
svm_tfidf.fit(X_train_tfidf_res, y_train_tfidf_res)

C:\Users\USER\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC()

In [79]:
## Making prediction on test_data (test_bow & test_tfidf)
X_test['svm_bow'] = svm_bow.predict(X_test_bow)
X_test['svm_tfidf'] = svm_tfidf.predict(X_test_tfidf)

In [80]:
X_test.head()

,id,username,location,total review,date of experience,text,text_without_stopwords,vader_on_text,vader_on_text_without_stopwords,mnb_bow,mnb_tfidf,logreg_bow,logreg_tfidf,svm_bow,svm_tfidf
45973,45973,Paul Philps,CA,3reviews,"August 31, 2013",I contacted a company about making 17 hockey j...,contacted company making 17 hockey jerseys ice...,positive,positive,negative,negative,negative,negative,negative,negative
22287,22287,TSANG David,FR,1review,"October 04, 2019","Received very quickly, I recommend for this se...","Received quickly , recommend seller , good pro...",positive,positive,positive,positive,positive,positive,positive,positive
34019,34019,Sunil,CH,1review,"July 20, 2018",I have been using Aliexpress for years. They d...,"using Aliexpress years . help , sellers want m...",positive,positive,positive,negative,positive,positive,positive,positive
16966,16966,Mikola Dmitruk,UA,1review,"February 19, 2020",Everything works flawlessly. Thank you.,Everything works flawlessly . Thank .,positive,positive,positive,positive,positive,positive,positive,positive
30467,30467,Joe Higashi,PH,1review,"January 10, 2019",their warm greetings makes I feel comfortable....,warm greetings makes feel comfortable . Ive st...,positive,positive,positive,neutral,positive,positive,positive,positive


In [81]:
# Adding the label to the test dataset (X_test) to compare with the vader_on_text, ader_on_text_without_stopwords, bow, tfidf.	
X_test['label'] = y_test

In [82]:
X_test.head()

,id,username,location,total review,date of experience,text,text_without_stopwords,vader_on_text,vader_on_text_without_stopwords,mnb_bow,mnb_tfidf,logreg_bow,logreg_tfidf,svm_bow,svm_tfidf,label
45973,45973,Paul Philps,CA,3reviews,"August 31, 2013",I contacted a company about making 17 hockey j...,contacted company making 17 hockey jerseys ice...,positive,positive,negative,negative,negative,negative,negative,negative,negative
22287,22287,TSANG David,FR,1review,"October 04, 2019","Received very quickly, I recommend for this se...","Received quickly , recommend seller , good pro...",positive,positive,positive,positive,positive,positive,positive,positive,positive
34019,34019,Sunil,CH,1review,"July 20, 2018",I have been using Aliexpress for years. They d...,"using Aliexpress years . help , sellers want m...",positive,positive,positive,negative,positive,positive,positive,positive,positive
16966,16966,Mikola Dmitruk,UA,1review,"February 19, 2020",Everything works flawlessly. Thank you.,Everything works flawlessly . Thank .,positive,positive,positive,positive,positive,positive,positive,positive,positive
30467,30467,Joe Higashi,PH,1review,"January 10, 2019",their warm greetings makes I feel comfortable....,warm greetings makes feel comfortable . Ive st...,positive,positive,positive,neutral,positive,positive,positive,positive,positive


##### Model Evaluation

##### 1. vader

In [83]:
# accuracy score on vader_on_text
vader_text_accuracy_score = accuracy_score(y_test, X_test['vader_on_text'])
vader_text_accuracy_score 

0.8157177880686461

In [84]:
# classification_report on vader_on_text
print(classification_report(y_test, X_test['vader_on_text']))

C:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

    negative       0.69      0.79      0.74      2301
     neutral       0.00      0.00      0.00       173
    positive       0.88      0.86      0.87      4868

    accuracy                           0.82      7342
   macro avg       0.53      0.55      0.54      7342
weighted avg       0.80      0.82      0.81      7342



C:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [85]:
vader_text_without_stopwords_accuracy_score = accuracy_score(y_test, X_test['vader_on_text_without_stopwords'])
vader_text_without_stopwords_accuracy_score

0.803868155815854

In [86]:
print(classification_report(y_test, X_test['vader_on_text_without_stopwords']))

C:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

    negative       0.70      0.71      0.71      2301
     neutral       0.00      0.00      0.00       173
    positive       0.85      0.88      0.86      4868

    accuracy                           0.80      7342
   macro avg       0.52      0.53      0.52      7342
weighted avg       0.78      0.80      0.79      7342



C:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##### 2. Naive Bayes

In [87]:
# accuracy score on bow
mnb_bow_score = accuracy_score(y_test, X_test['mnb_bow'])
mnb_bow_score

0.8952601470988831

In [88]:
# classification_report on bow
print(classification_report(y_test, X_test['mnb_bow']))

              precision    recall  f1-score   support

    negative       0.82      0.97      0.89      2301
     neutral       0.11      0.16      0.13       173
    positive       0.98      0.89      0.93      4868

    accuracy                           0.90      7342
   macro avg       0.64      0.67      0.65      7342
weighted avg       0.91      0.90      0.90      7342



In [89]:
# accuracy score on tfidf
mnb_tfidf_score = accuracy_score(y_test, X_test['mnb_tfidf'])
mnb_tfidf_score

0.8275674203214383

In [90]:
print(classification_report(y_test, X_test['mnb_tfidf']))

              precision    recall  f1-score   support

    negative       0.78      0.96      0.86      2301
     neutral       0.08      0.31      0.13       173
    positive       0.99      0.78      0.88      4868

    accuracy                           0.83      7342
   macro avg       0.62      0.69      0.62      7342
weighted avg       0.90      0.83      0.85      7342



##### 3. Logistic regression

In [91]:
logreg_bow_score = accuracy_score(y_test, X_test['logreg_bow'])
logreg_bow_score

0.9312176518659766

In [92]:
print(classification_report(y_test, X_test['logreg_bow']))

              precision    recall  f1-score   support

    negative       0.92      0.91      0.92      2301
     neutral       0.20      0.13      0.16       173
    positive       0.95      0.97      0.96      4868

    accuracy                           0.93      7342
   macro avg       0.69      0.67      0.68      7342
weighted avg       0.93      0.93      0.93      7342



In [93]:
print(classification_report(y_test, X_test['logreg_tfidf']))

              precision    recall  f1-score   support

    negative       0.89      0.95      0.92      2301
     neutral       0.24      0.22      0.23       173
    positive       0.97      0.94      0.96      4868

    accuracy                           0.93      7342
   macro avg       0.70      0.70      0.70      7342
weighted avg       0.93      0.93      0.93      7342



##### 4. Linear SVM

In [94]:
svm_bow_score = accuracy_score(y_test, X_test['svm_bow'])
svm_bow_score

0.9279487877962408

In [95]:
print(classification_report(y_test, X_test['svm_bow']))

              precision    recall  f1-score   support

    negative       0.92      0.90      0.91      2301
     neutral       0.15      0.08      0.10       173
    positive       0.94      0.97      0.96      4868

    accuracy                           0.93      7342
   macro avg       0.67      0.65      0.66      7342
weighted avg       0.92      0.93      0.92      7342



In [96]:
svm_tfidf_score = accuracy_score(y_test, X_test['svm_tfidf'])
svm_tfidf_score

0.9396622173794607

In [97]:
print(classification_report(y_test, X_test['svm_tfidf']))

              precision    recall  f1-score   support

    negative       0.90      0.96      0.93      2301
     neutral       0.16      0.03      0.05       173
    positive       0.96      0.96      0.96      4868

    accuracy                           0.94      7342
   macro avg       0.67      0.65      0.65      7342
weighted avg       0.93      0.94      0.93      7342



### Hyperparameter Tuning for Logistic Regression on TF-IDF

In [98]:
#from sklearn.feature_extraction.text import TfidfVectorizer
#from sklearn.linear_model import LogisticRegression
#from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline


X_train_text = X_train['text_without_stopwords']
X_test_text  = X_test['text_without_stopwords']

# vectorizer + resampler + classifier, all in one pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('ros',   RandomOverSampler(random_state=42)),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42)),
])

# just the handful of settings most likely to matter
params = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df':      [1, 3],
    'clf__C':             [0.1, 1, 10],
}

search = GridSearchCV(pipe, params, scoring='f1_macro', cv=3, n_jobs=-1)
search.fit(X_train_text, y_train)

print('Best params:', search.best_params_)
print('Best CV macro F1:', round(search.best_score_, 4))

best_model = search.best_estimator_
print(classification_report(y_test, best_model.predict(X_test_text)))

Best params: {'clf__C': 1, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2)}
Best CV macro F1: 0.6866
              precision    recall  f1-score   support

    negative       0.90      0.94      0.92      2301
     neutral       0.21      0.27      0.23       173
    positive       0.97      0.95      0.96      4868

    accuracy                           0.93      7342
   macro avg       0.70      0.72      0.71      7342
weighted avg       0.93      0.93      0.93      7342



##### Inference Script

In [99]:
def inference(text):
    """Takes raw review text, returns predicted sentiment."""
    filtered = remove_stopwords(text)
    return best_model.predict([filtered])[0]

In [100]:
example4 = "i love this product, it is very nice"

In [101]:
inference(example4)

'positive'

In [108]:
example3 = "I feel disgusted using this product and i will never buy it again"

In [109]:
inference(example3)

'negative'

#### Deployment

##### Save model to a file using python pickle

In [112]:
import joblib
joblib.dump(best_model, 'logreg_tfidf_tuned.joblib')

['logreg_tfidf_tuned.joblib']